# Lunden's Analysis of Movie Data

### Main RQ
What is the relationship between IMDB rating and income (domestic vs foreign vs internationl, gross vs %)

### Statistical Requirements
- Descriptive Statistics
- Inferential statistics
- Graphical Analysis
- Comparative Analysis
- Mulivariate Analysis
- Synthesis
- Documentation
- Reporting & Interpretation


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.simplefilter(action='ignore')
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

### Descriptive Statistics

In [3]:
movies = pd.read_csv('movies.csv')
print(movies.columns)

Index(['title', 'originalTitle', 'isAdult', 'runtimeMinutes', 'genres',
       'IMDBavgRating', 'numVotes', 'rank', 'worldwideGross', 'domesticGross',
       'domestic%', 'foreignGross', 'foreign%', 'year', 'originalLang',
       'productionCountries', 'main_genre', 'rating_category'],
      dtype='object')


In [8]:
movies.describe()

,isAdult,runtimeMinutes,IMDBavgRating,numVotes,rank,worldwideGross,domesticGross,domestic%,foreignGross,foreign%,year
count,7004.000000,7004.000000,7004.000000,7.004000e+03,7004.000000,7.004000e+03,7.004000e+03,7004.00000,7.004000e+03,7004.000000,7004.000000
mean,0.000714,104.654055,6.254326,8.545767e+04,101.490720,1.137223e+08,4.236133e+07,34.84486,7.136074e+07,65.153998,2012.621074
std,0.026711,28.926183,1.147439,1.812764e+05,56.298787,1.911045e+08,7.372391e+07,30.07744,1.271558e+08,30.076601,6.883668
min,0.000000,0.000000,1.200000,5.000000e+00,1.000000,4.693664e+06,0.000000e+00,0.00000,0.000000e+00,0.000000,2001.000000
25%,0.000000,92.000000,5.600000,4.117500e+02,54.000000,2.562145e+07,1.640742e+05,0.30000,1.480998e+07,42.900000,2007.000000
50%,0.000000,104.000000,6.400000,1.043750e+04,102.000000,4.841293e+07,1.830439e+07,36.60000,3.025607e+07,63.400000,2013.000000
75%,0.000000,119.000000,7.000000,9.189450e+04,150.000000,1.126836e+08,5.110049e+07,57.10000,7.030778e+07,99.600000,2019.000000
max,1.000000,700.000000,10.000000,2.985446e+06,200.000000,2.799439e+09,9.366622e+08,100.00000,1.993811e+09,100.000000,2024.000000


In [5]:
def print_median(df):
    '''Prints the median of columns with numeric data types'''

    for column in df.select_dtypes(include=[np.number]).columns:
        print(f'{column}: {df[column].median()}')

print_median(movies)

isAdult: 0.0
runtimeMinutes: 104.0
IMDBavgRating: 6.4
numVotes: 10437.5
rank: 102.0
worldwideGross: 48412933.0
domesticGross: 18304386.5
domestic%: 36.6
foreignGross: 30256068.0
foreign%: 63.4
year: 2013.0


In [17]:
def print_grouped_stats(df):
    for column in df.select_dtypes(include=[np.number]).columns:
        print(f"{column} stats by rating category")
        print(f"{df.groupby('rating_category')[column].describe()}\n")

print_grouped_stats(movies)

isAdult stats by rating category
                  count      mean       std  min  25%  50%  75%  max
rating_category                                                     
High             1728.0  0.000579  0.024056  0.0  0.0  0.0  0.0  1.0
Low               310.0  0.000000  0.000000  0.0  0.0  0.0  0.0  0.0
Medium           4966.0  0.000805  0.028372  0.0  0.0  0.0  0.0  1.0

runtimeMinutes stats by rating category
                  count        mean        std  min   25%    50%    75%    max
rating_category                                                               
High             1728.0  110.563657  33.633029  0.0  96.0  114.0  130.0  280.0
Low               310.0   91.416129  32.843126  0.0  83.0   92.0  105.0  163.0
Medium           4966.0  103.424084  26.362317  0.0  92.0  103.0  116.0  700.0

IMDBavgRating stats by rating category
                  count      mean       std  min  25%  50%  75%   max
rating_category                                                      
High  

### Interpretation of Descriptive Statistics

The first generated dataframe presents general descriptive statistics of the movies data, displaying useful metrics such as mean, standard devation, and the range of values for each numeric feature of the data. I also created print_median() functions to display the median values of each numeric feature. I then grouped the data by rating category (high/medium/low) and compared the descriptive statistics of this dataframe to the original dataframe.

*isAdult*\
This feature is a boolean representation of the adult movie categorization. The mean value is 0.00071, which  means not many movies in this data are adult films. When grouped by rating category, I find that there are adult movies in the high and medium rated categories and not the low rated category.

*runtimeMinutes*\
The mean runtime of movies in this data is 104.7 minutes and the standard deviation is 28.9 minutes. This tells use that there is a large range of runtimes. High rated movies have a higher mean runtime than medium, and medium rated movies higher than low rated movies. There could be a correlation to find here.

*IMDBavgRating*\
The mean rating 6.3 and the standard deviation is 1.1. From this I can determine that a majority of movies are rated between 5 and 7. The mean average ratings for high rated movies and medium rated movies are close to the full sample mean.

*Income metrics*\
The gross income means for films (worldwide, foreign, and domestic) are in the millions, and the standard deviation are just as large. There is significant variation in these metrics. The median values for each type of gross income are significantly lower than the mean values, suggesting that there is a small number of movies that are making significantly more money than the others. High rated movies have higher mean gross income levels than medium and low rated movies. An interesting finding is that medium and low rated movies have a similar mean foreign gross income level.

### Inferential Statistics
1. Is there a linear relationship between IMDB rating and gross income? What if I separate by rating category?
2. Do highly rated movies generate more revenue than medium rated movies?
3. Do medium rated movies generate more revenue than low rated movies?

*Testing Methods*
- ANOVA
- Regression Analysis
- Hypothesis Testing
